In [ ]:
import pandas as pd
from pathlib import Path

data_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "raw" / "swat"

df_normal = pd.read_csv(data_dir / "Normal.csv", encoding="utf-8", low_memory=False)
df_attack = pd.read_csv(data_dir / "Attack.csv", encoding="utf-8", low_memory=False)

print("Normal shape:", df_normal.shape)
print("Attack shape:", df_attack.shape)

print("\nNormal columns:")
print(df_normal.columns.tolist())

print("\nAttack columns:")
print(df_attack.columns.tolist())

print("\nNormal head:")
display(df_normal.head())

print("\nAttack head:")
display(df_attack.head())

In [ ]:
import pandas as pd
from pathlib import Path

data_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "raw" / "swat"
out_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed" / "swat"
out_dir.mkdir(parents=True, exist_ok=True)

df_normal = pd.read_csv(data_dir / "Normal.csv", low_memory=False)
df_attack = pd.read_csv(data_dir / "Attack.csv", low_memory=False)

def clean_swat(df):
    df = df.copy()
    
    # 1) 清洗列名空格
    df.columns = df.columns.str.strip()
    
    # 2) 时间列转 datetime
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], dayfirst=True, errors="coerce")
    
    # 3) 标签列转 0/1
    df["label"] = df["Normal/Attack"].map({"Normal": 0, "Attack": 1})
    
    return df

df_normal_clean = clean_swat(df_normal)
df_attack_clean = clean_swat(df_attack)

print(df_normal_clean.columns.tolist())
print(df_attack_clean.columns.tolist())

print(df_normal_clean[["Timestamp", "Normal/Attack", "label"]].head())
print(df_attack_clean[["Timestamp", "Normal/Attack", "label"]].head())

df_normal_clean.to_csv(out_dir / "normal_clean.csv", index=False)
df_attack_clean.to_csv(out_dir / "attack_clean.csv", index=False)

print("saved:", out_dir / "normal_clean.csv")
print("saved:", out_dir / "attack_clean.csv")

In [ ]:
import pandas as pd
from pathlib import Path

proc_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed" / "swat"

df_normal = pd.read_csv(proc_dir / "normal_clean.csv", parse_dates=["Timestamp"])
df_attack = pd.read_csv(proc_dir / "attack_clean.csv", parse_dates=["Timestamp"])

df_merged = pd.concat([df_normal, df_attack], axis=0, ignore_index=True)
df_merged = df_merged.sort_values("Timestamp").reset_index(drop=True)

print("merged shape:", df_merged.shape)
print(df_merged[["Timestamp", "Normal/Attack", "label"]].head())
print(df_merged[["Timestamp", "Normal/Attack", "label"]].tail())

df_merged.to_csv(proc_dir / "merged.csv", index=False)
print("saved:", proc_dir / "merged.csv")

In [ ]:
import pandas as pd
from pathlib import Path

proc_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed" / "swat"

df = pd.read_csv(proc_dir / "merged.csv")
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")

print("before:", df.shape)
print("NaT count:", df["Timestamp"].isna().sum())

df = df.dropna(subset=["Timestamp"]).sort_values("Timestamp").reset_index(drop=True)

print("after:", df.shape)
print(df[["Timestamp", "Normal/Attack", "label"]].head())
print(df[["Timestamp", "Normal/Attack", "label"]].tail())

df.to_csv(proc_dir / "merged_clean_time.csv", index=False)
print("saved:", proc_dir / "merged_clean_time.csv")

In [ ]:
import pandas as pd
from pathlib import Path

proc_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed" / "swat"

df = pd.read_csv(proc_dir / "merged_clean_time.csv", parse_dates=["Timestamp"])

exclude_cols = ["Timestamp", "Normal/Attack", "label"]
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols].copy()
y = df["label"].copy()

print("特征数:", len(feature_cols))
print("前10个特征名:", feature_cols[:10])
print("X shape:", X.shape)
print("y shape:", y.shape)

X.to_csv(proc_dir / "X.csv", index=False)
y.to_csv(proc_dir / "y.csv", index=False)

print("saved:", proc_dir / "X.csv")
print("saved:", proc_dir / "y.csv")

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import joblib

proc_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed" / "swat"

X = pd.read_csv(proc_dir / "X.csv")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

df_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("scaled shape:", df_scaled.shape)
print(df_scaled.head())

df_scaled.to_csv(proc_dir / "X_scaled.csv", index=False)
joblib.dump(scaler, proc_dir / "scaler.pkl")

print("saved:", proc_dir / "X_scaled.csv")
print("saved:", proc_dir / "scaler.pkl")

In [ ]:
import pandas as pd
from pathlib import Path

proc_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed" / "swat"

X = pd.read_csv(proc_dir / "X.csv")

nan_count = X.isna().sum()
nan_cols = nan_count[nan_count > 0].sort_values(ascending=False)

print("含 NaN 的列数:", len(nan_cols))
print(nan_cols)

# 再看这些列是不是数值列
print("\n这些列的数据类型：")
print(X[nan_cols.index].dtypes)

In [ ]:
import pandas as pd
from pathlib import Path

proc_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed" / "swat"

df_normal = pd.read_csv(proc_dir / "normal_clean.csv")
df_attack = pd.read_csv(proc_dir / "attack_clean.csv")

bad_cols = ["MV101", "AIT201", "MV201", "P201", "P202", "P204", "MV303"]

print("Normal:")
print(df_normal[bad_cols].isna().sum())

print("\nAttack:")
print(df_attack[bad_cols].isna().sum())

In [ ]:
import pandas as pd
from pathlib import Path

raw_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "raw" / "swat"

# 用字符串方式读取，避免自动转 NaN
df_normal_raw = pd.read_csv(raw_dir / "Normal.csv", dtype=str, low_memory=False)
df_normal_raw.columns = df_normal_raw.columns.str.strip()

bad_cols = ["MV101", "AIT201", "MV201", "P201", "P202", "P204", "MV303"]

for col in bad_cols:
    s = df_normal_raw[col].astype(str).str.strip()
    numeric_s = pd.to_numeric(s, errors="coerce")
    
    bad_mask = numeric_s.isna()
    print(f"\n===== {col} =====")
    print("非数值个数:", bad_mask.sum())
    print("前20个异常原始值:")
    print(s[bad_mask].drop_duplicates().head(20).tolist())

In [ ]:
import pandas as pd
from pathlib import Path

raw_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "raw" / "swat"

df_normal_raw = pd.read_csv(
    raw_dir / "Normal.csv",
    dtype=str,
    low_memory=False,
    keep_default_na=False
)
df_normal_raw.columns = df_normal_raw.columns.str.strip()

bad_cols = ["MV101", "AIT201", "MV201", "P201", "P202", "P204", "MV303"]

for col in bad_cols:
    s = df_normal_raw[col].astype(str)
    empty_mask = s.str.strip() == ""
    
    print(f"\n===== {col} =====")
    print("空字符串个数:", empty_mask.sum())
    print("前10个唯一值:", s.drop_duplicates().head(10).tolist())

In [ ]:
import pandas as pd
from pathlib import Path

proc_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed" / "swat"

df = pd.read_csv(proc_dir / "merged_clean_time.csv", parse_dates=["Timestamp"])

bad_cols = ["MV101", "AIT201", "MV201", "P201", "P202", "P204", "MV303"]

print("修复前 NaN：")
print(df[bad_cols].isna().sum())

# 前向填充，再后向填充
df[bad_cols] = df[bad_cols].ffill().bfill()

print("\n修复后 NaN：")
print(df[bad_cols].isna().sum())

df.to_csv(proc_dir / "merged_filled.csv", index=False)
print("\nsaved:", proc_dir / "merged_filled.csv")

In [ ]:
import pandas as pd
from pathlib import Path

proc_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed" / "swat"

df = pd.read_csv(proc_dir / "merged_filled.csv", parse_dates=["Timestamp"])

exclude_cols = ["Timestamp", "Normal/Attack", "label"]
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols].copy()
y = df["label"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X 中 NaN 总数:", X.isna().sum().sum())

X.to_csv(proc_dir / "X_filled.csv", index=False)
y.to_csv(proc_dir / "y_filled.csv", index=False)

print("saved:", proc_dir / "X_filled.csv")
print("saved:", proc_dir / "y_filled.csv")

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import joblib

proc_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed" / "swat"

X = pd.read_csv(proc_dir / "X_filled.csv")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

df_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("scaled shape:", df_scaled.shape)
print("scaled NaN 总数:", df_scaled.isna().sum().sum())

df_scaled.to_csv(proc_dir / "X_filled_scaled.csv", index=False)
joblib.dump(scaler, proc_dir / "scaler_filled.pkl")

print("saved:", proc_dir / "X_filled_scaled.csv")
print("saved:", proc_dir / "scaler_filled.pkl")

In [ ]:
import pandas as pd
from pathlib import Path

proc_dir = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "data" / "processed" / "swat"

X_scaled = pd.read_csv(proc_dir / "X_filled_scaled.csv")

df_binary = (X_scaled > 0).astype(int)

print("binary shape:", df_binary.shape)
print("取值检查:", sorted(df_binary.stack().unique().tolist()))
print(df_binary.head())

df_binary.to_csv(proc_dir / "X_filled_binary.csv", index=False)
print("saved:", proc_dir / "X_filled_binary.csv")